# Challenge 1: K-Means Clustering for Images (MNIST)
**Numerical Recipes for Machine Learning**

This notebook contains all implementations and experiments for Challenge 1.
Each section corresponds to one assignment task.


## 0. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
import os, time, warnings
warnings.filterwarnings('ignore')
from sklearn.datasets import fetch_openml

os.makedirs('plots', exist_ok=True)
print("Imports OK")


## 1. Core K-Means Routines

In [ ]:
"""
Core K-means routines for the MNIST clustering challenge.
All functions are implemented from scratch using only numpy.
"""

import numpy as np


# ---------------------------------------------------------------------------
# Distance computation
# ---------------------------------------------------------------------------

def compute_distance_matrix(X, C):
    """
    Compute squared Euclidean distance matrix between data points and centroids.

    Uses the identity ||x - c||^2 = ||x||^2 - 2 x·c + ||c||^2
    to avoid an explicit double for-loop.

    Parameters
    ----------
    X : ndarray, shape (N, d)
    C : ndarray, shape (K, d)

    Returns
    -------
    D : ndarray, shape (N, K)  — squared distances
    """
    # ||x||^2  shape (N,)
    x_sq = np.sum(X ** 2, axis=1, keepdims=True)   # (N, 1)
    # ||c||^2  shape (K,)
    c_sq = np.sum(C ** 2, axis=1, keepdims=True).T  # (1, K)
    # cross term  shape (N, K)
    cross = X @ C.T                                  # (N, K)
    D = x_sq - 2.0 * cross + c_sq
    # Numerical noise can produce tiny negatives; clamp to 0
    np.maximum(D, 0.0, out=D)
    return D


def compute_weighted_distance_matrix(X, C, w):
    """
    Compute weighted squared distance matrix.
    d_w(x, c)^2 = sum_j w_j (x_j - c_j)^2

    Parameters
    ----------
    X : ndarray, shape (N, d)
    C : ndarray, shape (K, d)
    w : ndarray, shape (d,)  — non-negative weights

    Returns
    -------
    D : ndarray, shape (N, K)
    """
    Xw = X * np.sqrt(w)   # (N, d)
    Cw = C * np.sqrt(w)   # (K, d)
    return compute_distance_matrix(Xw, Cw)


# ---------------------------------------------------------------------------
# Assignment step
# ---------------------------------------------------------------------------

def assign_clusters(X, C, w=None):
    """
    Assign each point to its nearest centroid.

    Parameters
    ----------
    X : ndarray, shape (N, d)
    C : ndarray, shape (K, d)
    w : ndarray, shape (d,) or None — if given, use weighted distance

    Returns
    -------
    labels : ndarray, shape (N,)  — cluster index for each point
    """
    if w is None:
        D = compute_distance_matrix(X, C)
    else:
        D = compute_weighted_distance_matrix(X, C, w)
    return np.argmin(D, axis=1)


# ---------------------------------------------------------------------------
# Update step
# ---------------------------------------------------------------------------

def update_centroids(X, labels, K):
    """
    Compute new centroids as the mean of assigned points.

    If a cluster becomes empty, its centroid is kept unchanged
    (handled by the caller who passes in old centroids).

    Parameters
    ----------
    X      : ndarray, shape (N, d)
    labels : ndarray, shape (N,)
    K      : int

    Returns
    -------
    C_new : ndarray, shape (K, d)
    counts: ndarray, shape (K,) — number of points in each cluster
    """
    d = X.shape[1]
    C_new = np.zeros((K, d), dtype=np.float64)
    counts = np.zeros(K, dtype=np.int64)
    for k in range(K):
        mask = labels == k
        counts[k] = mask.sum()
        if counts[k] > 0:
            C_new[k] = X[mask].mean(axis=0)
    return C_new, counts


# ---------------------------------------------------------------------------
# Objective function
# ---------------------------------------------------------------------------

def compute_objective(X, labels, C):
    """
    Compute the K-means within-cluster sum of squared distances.

    Parameters
    ----------
    X      : ndarray, shape (N, d)
    labels : ndarray, shape (N,)
    C      : ndarray, shape (K, d)

    Returns
    -------
    obj : float
    """
    # Vectorized: subtract each point's assigned centroid, then sum squared norms
    diff = X - C[labels]          # (N, d)
    return float(np.sum(diff ** 2))


# ---------------------------------------------------------------------------
# Initialization
# ---------------------------------------------------------------------------

def _init_random(X, K, rng):
    """Random initialization: pick K distinct points uniformly."""
    idx = rng.choice(len(X), size=K, replace=False)
    return X[idx].copy()


def _init_kmeanspp(X, K, rng):
    """
    K-means++ initialization.
    First centroid: uniform random.
    Each subsequent centroid: sample proportional to squared distance
    from the nearest already-chosen centroid.
    """
    N = len(X)
    idx0 = rng.integers(0, N)
    centroids = [X[idx0].copy()]

    for _ in range(1, K):
        C = np.array(centroids)
        D = compute_distance_matrix(X, C)           # (N, k_so_far)
        min_dist = D.min(axis=1)                    # (N,)
        probs = min_dist / min_dist.sum()
        next_idx = rng.choice(N, p=probs)
        centroids.append(X[next_idx].copy())

    return np.array(centroids)


# ---------------------------------------------------------------------------
# Main K-means
# ---------------------------------------------------------------------------

def kmeans(X, K, max_iter=300, tol=1e-4, n_init=5, init='kmeans++',
           w=None, random_state=42, verbose=False):
    """
    K-means clustering with multiple restarts.

    Parameters
    ----------
    X            : ndarray, shape (N, d)
    K            : int
    max_iter     : int
    tol          : float  — stop if objective improves by less than tol (relative)
    n_init       : int    — number of random restarts; best result is returned
    init         : 'random' or 'kmeans++'
    w            : ndarray, shape (d,) or None — feature weights
    random_state : int
    verbose      : bool

    Returns
    -------
    best_labels    : ndarray, shape (N,)
    best_centroids : ndarray, shape (K, d)
    best_obj       : float
    best_history   : list of floats — objective per iteration (best run)
    """
    rng = np.random.default_rng(random_state)

    best_obj = np.inf
    best_labels = None
    best_centroids = None
    best_history = None

    for run in range(n_init):
        # --- Initialization ---
        if init == 'kmeans++':
            C = _init_kmeanspp(X, K, rng)
        else:
            C = _init_random(X, K, rng)

        history = []

        for it in range(max_iter):
            # Assignment
            labels = assign_clusters(X, C, w=w)

            # New centroids; keep old ones for empty clusters
            C_new, counts = update_centroids(X, labels, K)
            empty = counts == 0
            C_new[empty] = C[empty]

            obj = compute_objective(X, labels, C_new)
            history.append(obj)

            if verbose:
                print(f"  Run {run+1}, iter {it+1}: obj={obj:.4f}")

            # Check for empty-cluster reassignment
            if empty.any():
                # Re-assign empty centroid to a random point
                for k in np.where(empty)[0]:
                    C_new[k] = X[rng.integers(0, len(X))]

            # Convergence check (after first iteration)
            if it > 0 and abs(history[-2] - obj) / (abs(history[-2]) + 1e-12) < tol:
                break

            C = C_new

        if obj < best_obj:
            best_obj = obj
            best_labels = labels.copy()
            best_centroids = C_new.copy()
            best_history = history

    return best_labels, best_centroids, best_obj, best_history


# ---------------------------------------------------------------------------
# Evaluation metrics
# ---------------------------------------------------------------------------

def cluster_purity(labels, true_labels):
    """
    Compute cluster purity: fraction of points that are in the majority
    class of their assigned cluster.
    """
    K = labels.max() + 1
    total = len(labels)
    correct = 0
    for k in range(K):
        mask = labels == k
        if mask.sum() == 0:
            continue
        counts = np.bincount(true_labels[mask])
        correct += counts.max()
    return correct / total


def cluster_entropy(labels, true_labels):
    """
    Mean entropy of label distribution inside each cluster.
    """
    K = labels.max() + 1
    entropies = []
    for k in range(K):
        mask = labels == k
        if mask.sum() == 0:
            continue
        counts = np.bincount(true_labels[mask])
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        entropies.append(-np.sum(probs * np.log(probs + 1e-12)))
    return float(np.mean(entropies))


def normalized_mutual_information(labels, true_labels):
    """
    Compute normalized mutual information (NMI) between cluster labels and true labels.
    NMI = 2 * MI(labels, true_labels) / (H(labels) + H(true_labels))
    """
    N = len(labels)
    K = labels.max() + 1
    C_classes = true_labels.max() + 1

    # Joint distribution
    joint = np.zeros((K, C_classes))
    for k, c in zip(labels, true_labels):
        joint[k, c] += 1
    joint /= N

    # Marginals
    p_k = joint.sum(axis=1, keepdims=True)   # (K, 1)
    p_c = joint.sum(axis=0, keepdims=True)   # (1, C)

    # MI
    mask = joint > 0
    mi = np.sum(joint[mask] * np.log(joint[mask] / (p_k * p_c + 1e-300)[mask]))

    # Entropies
    h_k = -np.sum(p_k[p_k > 0] * np.log(p_k[p_k > 0]))
    h_c = -np.sum(p_c[p_c > 0] * np.log(p_c[p_c > 0]))

    if h_k + h_c < 1e-12:
        return 0.0
    return float(2.0 * mi / (h_k + h_c))


def silhouette_score_fast(X, labels, sample_size=2000, random_state=42):
    """
    Approximate silhouette score using a random subsample.

    s(i) = (b(i) - a(i)) / max(a(i), b(i))
    a(i) = mean intra-cluster distance
    b(i) = mean distance to nearest other cluster
    """
    rng = np.random.default_rng(random_state)
    N = len(X)
    idx = rng.choice(N, size=min(sample_size, N), replace=False)
    Xs = X[idx]
    ls = labels[idx]

    K = labels.max() + 1
    scores = []
    for i in range(len(Xs)):
        k = ls[i]
        intra_mask = ls == k
        inter_masks = [ls == j for j in range(K) if j != k]

        if intra_mask.sum() > 1:
            a = np.mean(np.sqrt(np.sum((Xs[intra_mask] - Xs[i]) ** 2, axis=1)))
        else:
            a = 0.0

        b_vals = []
        for m in inter_masks:
            if m.sum() > 0:
                b_vals.append(np.mean(np.sqrt(np.sum((Xs[m] - Xs[i]) ** 2, axis=1))))
        b = min(b_vals) if b_vals else 0.0

        denom = max(a, b)
        s = (b - a) / denom if denom > 0 else 0.0
        scores.append(s)

    return float(np.mean(scores))


def train_test_cluster_evaluation(X_train, X_test, y_train, y_test,
                                  K, n_init=3, init='kmeans++',
                                  w=None, random_state=42):
    """
    Fit K-means on train set, evaluate on both train and test.

    Returns dict with train/test objectives, purity, NMI, entropy.
    """
    labels_tr, centroids, obj_tr, hist = kmeans(
        X_train, K, n_init=n_init, init=init, w=w, random_state=random_state
    )
    labels_te = assign_clusters(X_test, centroids, w=w)
    obj_te = compute_objective(X_test, labels_te, centroids)

    return {
        'train_obj': obj_tr,
        'test_obj': obj_te,
        'train_purity': cluster_purity(labels_tr, y_train),
        'test_purity': cluster_purity(labels_te, y_test),
        'train_nmi': normalized_mutual_information(labels_tr, y_train),
        'test_nmi': normalized_mutual_information(labels_te, y_test),
        'train_entropy': cluster_entropy(labels_tr, y_train),
        'test_entropy': cluster_entropy(labels_te, y_test),
        'centroids': centroids,
        'train_labels': labels_tr,
        'test_labels': labels_te,
        'history': hist,
    }


## 2. Load MNIST Dataset

In [ ]:
print("Loading MNIST (may take a moment)...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_all = mnist.data.astype(np.float64) / 255.0  # Normalise to [0,1]
y_all = mnist.target.astype(int)

X = X_all[:60000]   # 60 000 training images
y = y_all[:60000]

print(f"Dataset shape: {X.shape}, labels: {np.unique(y)}")


## 3. Task 1 — Basic K-Means on MNIST

**Objectives:**
- Flatten 28×28 images to 784-dim vectors, normalize to [0,1]
- Run K-means for K ∈ {5, 10, 15, 20, 30}
- Compare random init vs K-means++ init
- Track objective per iteration
- Visualize centroids and nearest images


In [ ]:
"""
Task 1 — Basic K-means on MNIST
Runs experiments, saves all plots to plots/
"""

import time



# ============================================================
# 1. Load MNIST
# ============================================================
# (Data already loaded above)
PLOTS = 'plots'

# ============================================================
# 2. Convergence plot — objective vs iteration for K=10
# ============================================================
print("\n[Task 1] Objective vs iteration for K=10 (random vs kmeans++)")

K_demo = 10

labels_rand, C_rand, obj_rand, hist_rand = kmeans(
    X, K_demo, max_iter=100, n_init=1, init='random', random_state=0
)
labels_pp, C_pp, obj_pp, hist_pp = kmeans(
    X, K_demo, max_iter=100, n_init=1, init='kmeans++', random_state=0
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_rand, label=f'Random init (final={obj_rand:.2e})', marker='o', markersize=3)
ax.plot(hist_pp, label=f'K-means++ init (final={obj_pp:.2e})', marker='s', markersize=3)
ax.set_xlabel('Iteration')
ax.set_ylabel('Within-cluster SSE')
ax.set_title('K-means convergence (K=10, MNIST)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1_convergence.png', dpi=120)
plt.close()
print("  Saved task1_convergence.png")

# ============================================================
# 3. Run K-means for multiple K values
# ============================================================
K_values = [5, 10, 15, 20, 30]
results = {}

for K in K_values:
    print(f"\n  K={K} ...", end=' ', flush=True)
    t0 = time.time()
    labels, centroids, obj, hist = kmeans(
        X, K, max_iter=150, n_init=3, init='kmeans++', random_state=42
    )
    dt = time.time() - t0
    purity = cluster_purity(labels, y)
    nmi = normalized_mutual_information(labels, y)
    results[K] = dict(labels=labels, centroids=centroids, obj=obj,
                      hist=hist, purity=purity, nmi=nmi, time=dt)
    print(f"obj={obj:.3e}, purity={purity:.3f}, NMI={nmi:.3f}, t={dt:.1f}s")

# ============================================================
# 4. Visualize centroids for K=10
# ============================================================
print("\n[Task 1] Visualizing centroids for K=10")
C10 = results[10]['centroids']

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.flat):
    ax.imshow(C10[k].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Centroid {k}')
    ax.axis('off')
plt.suptitle('K-means Centroids (K=10, pixel domain)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1_centroids_k10.png', dpi=120)
plt.close()
print("  Saved task1_centroids_k10.png")

# ============================================================
# 5. Show nearest images to each centroid (K=10)
# ============================================================
print("[Task 1] Nearest images to centroids for K=10")
labels10 = results[10]['labels']
n_show = 5   # images per cluster

fig, axes = plt.subplots(10, n_show + 1, figsize=(2 * (n_show + 1), 21))
D_full = np.sum((X - C10[labels10]) ** 2, axis=1)  # distance to own centroid

for k in range(10):
    mask = np.where(labels10 == k)[0]
    # Sort by distance to centroid k
    dists_k = np.sum((X[mask] - C10[k]) ** 2, axis=1)
    sorted_idx = mask[np.argsort(dists_k)]
    nearest = sorted_idx[:n_show]

    # First column: centroid
    axes[k, 0].imshow(C10[k].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[k, 0].set_title('centroid', fontsize=7)
    axes[k, 0].axis('off')
    # Remaining: nearest images
    for j, idx in enumerate(nearest):
        axes[k, j + 1].imshow(X[idx].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        axes[k, j + 1].set_title(f'digit {y[idx]}', fontsize=7)
        axes[k, j + 1].axis('off')

plt.suptitle('Centroids and nearest images (K=10)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1_nearest_images_k10.png', dpi=100)
plt.close()
print("  Saved task1_nearest_images_k10.png")

# ============================================================
# 6. Sensitivity to initialization: multiple runs, K=10
# ============================================================
print("\n[Task 1] Sensitivity to initialization")
n_runs = 8
objs_random = []
objs_pp = []

for run in range(n_runs):
    _, _, o_r, _ = kmeans(X, 10, max_iter=100, n_init=1,
                          init='random', random_state=run * 100)
    _, _, o_p, _ = kmeans(X, 10, max_iter=100, n_init=1,
                          init='kmeans++', random_state=run * 100)
    objs_random.append(o_r)
    objs_pp.append(o_p)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(range(n_runs), objs_random, label='Random init', s=60, zorder=3)
ax.scatter(range(n_runs), objs_pp, label='K-means++ init', marker='s', s=60, zorder=3)
ax.axhline(np.mean(objs_random), color='C0', linestyle='--', alpha=0.5)
ax.axhline(np.mean(objs_pp), color='C1', linestyle='--', alpha=0.5)
ax.set_xlabel('Run index')
ax.set_ylabel('Final objective (SSE)')
ax.set_title('Initialization sensitivity (K=10, single restart each)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1_init_sensitivity.png', dpi=120)
plt.close()
print("  Saved task1_init_sensitivity.png")

# ============================================================
# 7. Centroid gallery for K=5 and K=20
# ============================================================
for K in [5, 20]:
    C = results[K]['centroids']
    cols = min(K, 10)
    rows = (K + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2 * rows))
    axes = np.array(axes).reshape(-1)
    for k in range(K):
        axes[k].imshow(C[k].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        axes[k].set_title(f'{k}', fontsize=7)
        axes[k].axis('off')
    for k in range(K, len(axes)):
        axes[k].axis('off')
    plt.suptitle(f'Centroids (K={K})', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{PLOTS}/task1_centroids_k{K}.png', dpi=120)
    plt.close()
    print(f"  Saved task1_centroids_k{K}.png")

# ============================================================
# 8. Summary table
# ============================================================
print("\n=== Summary ===")
print(f"{'K':>4} {'Objective':>14} {'Purity':>8} {'NMI':>8} {'Time(s)':>8}")
for K in K_values:
    r = results[K]
    print(f"{K:>4} {r['obj']:>14.3e} {r['purity']:>8.4f} {r['nmi']:>8.4f} {r['time']:>8.1f}")

# Save results for other tasks
np.save('plots/task1_results_summary.npy', {
    K: {'obj': results[K]['obj'],
        'purity': results[K]['purity'],
        'nmi': results[K]['nmi']}
    for K in K_values
})

print("\nTask 1 complete.")


## 4. Task 1a — How to Choose K?

Methods: Elbow curve, Silhouette score, Cluster purity (external), Stability.


In [ ]:
"""
Task 1a — How to choose K?
Elbow curve, silhouette, purity, stability.
"""



# (Data already loaded above)
PLOTS = 'plots'
# Subset for silhouette (expensive O(n^2) per cluster)
sil_size = 3000
rng = np.random.default_rng(0)
sil_idx = rng.choice(len(X), sil_size, replace=False)
X_sil = X[sil_idx]
y_sil = y[sil_idx]

K_values = [2, 5, 8, 10, 12, 15, 20, 25, 30]

objectives   = []
purities     = []
nmis         = []
silhouettes  = []
stab_stds    = []

print("\n[Task 1a] Scanning K values...")
for K in K_values:
    print(f"  K={K}...", end=' ', flush=True)

    # Main run (3 restarts)
    labels, centroids, obj, hist = kmeans(
        X, K, max_iter=150, n_init=3, init='kmeans++', random_state=42
    )
    objectives.append(obj)
    purities.append(cluster_purity(labels, y))
    nmis.append(normalized_mutual_information(labels, y))

    # Silhouette on subset
    labels_sil = assign_clusters(X_sil, centroids)
    sil = silhouette_score_fast(X_sil, labels_sil, sample_size=sil_size)
    silhouettes.append(sil)

    # Stability: run 4 independent restarts, measure std of objectives
    run_objs = []
    for seed in range(4):
        _, _, o, _ = kmeans(X, K, max_iter=100, n_init=1,
                            init='kmeans++', random_state=seed * 7)
        run_objs.append(o)
    stab_stds.append(np.std(run_objs))

    print(f"obj={obj:.3e}, purity={purities[-1]:.3f}, "
          f"NMI={nmis[-1]:.3f}, sil={sil:.3f}, std={stab_stds[-1]:.2e}")

# ============================================================
# Plots
# ============================================================

# --- Elbow curve ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_values, objectives, 'o-', color='steelblue')
ax.set_xlabel('K (number of clusters)')
ax.set_ylabel('Within-cluster SSE')
ax.set_title('Elbow Curve')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1a_elbow.png', dpi=120)
plt.close()
print("\nSaved task1a_elbow.png")

# --- Silhouette score ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_values, silhouettes, 's-', color='darkorange')
ax.set_xlabel('K')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette Score vs K')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1a_silhouette.png', dpi=120)
plt.close()
print("Saved task1a_silhouette.png")

# --- Purity & NMI ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_values, purities, 'o-', label='Purity', color='green')
ax.plot(K_values, nmis, 's--', label='NMI', color='purple')
ax.set_xlabel('K')
ax.set_ylabel('Score')
ax.set_title('Cluster Purity and NMI vs K')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1a_purity_nmi.png', dpi=120)
plt.close()
print("Saved task1a_purity_nmi.png")

# --- Stability (std of objective across runs) ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_values, stab_stds, 'd-', color='crimson')
ax.set_xlabel('K')
ax.set_ylabel('Std of objective (4 runs)')
ax.set_title('Stability vs K')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1a_stability.png', dpi=120)
plt.close()
print("Saved task1a_stability.png")

# --- Combined 4-panel figure ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(K_values, objectives, 'o-', color='steelblue')
axes[0, 0].set_title('Elbow Curve (SSE)')
axes[0, 0].set_xlabel('K'); axes[0, 0].set_ylabel('SSE')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(K_values, silhouettes, 's-', color='darkorange')
axes[0, 1].set_title('Silhouette Score')
axes[0, 1].set_xlabel('K'); axes[0, 1].set_ylabel('Score')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(K_values, purities, 'o-', label='Purity', color='green')
axes[1, 0].plot(K_values, nmis, 's--', label='NMI', color='purple')
axes[1, 0].set_title('Purity & NMI (external, uses labels)')
axes[1, 0].set_xlabel('K'); axes[1, 0].set_ylabel('Score')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(K_values, stab_stds, 'd-', color='crimson')
axes[1, 1].set_title('Stability (std of objective, 4 runs)')
axes[1, 1].set_xlabel('K'); axes[1, 1].set_ylabel('Std(SSE)')
axes[1, 1].grid(alpha=0.3)

plt.suptitle('K Selection Methods — MNIST', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task1a_combined.png', dpi=120)
plt.close()
print("Saved task1a_combined.png")

print("\nTask 1a complete.")


## 5. Task 2 — K-Means in the DFT Domain with Feature Weighting

Represents each image using 2D DFT log-magnitude features.
Implements weighted distance with several frequency weight masks.


In [ ]:
"""
Task 2 — K-means in the DFT domain with feature weighting.
Task 2a — Optimizing feature weights.
"""



# (Data already loaded above)
PLOTS = 'plots'
# ============================================================
# 2. DFT feature extraction
# ============================================================

def dft_features(images, mode='magnitude'):
    """
    Compute 2D DFT of each image and return a feature matrix.

    Parameters
    ----------
    images : ndarray, shape (N, 784) — flattened 28x28 images
    mode   : 'magnitude' | 'phase' | 'both'

    Returns
    -------
    features : ndarray, shape (N, d)
    """
    N = len(images)
    imgs = images.reshape(N, 28, 28)
    F = np.fft.fft2(imgs)   # (N, 28, 28) complex

    if mode == 'magnitude':
        # log(1 + |F|) for better dynamic range
        feats = np.log1p(np.abs(F)).reshape(N, -1)
    elif mode == 'phase':
        feats = np.angle(F).reshape(N, -1)
    elif mode == 'both':
        mag = np.log1p(np.abs(F)).reshape(N, -1)
        phase = np.angle(F).reshape(N, -1)
        feats = np.concatenate([mag, phase], axis=1)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    return feats.astype(np.float64)


def build_radial_weights(shape=(28, 28), mode='low', sigma=7.0):
    """
    Build frequency-domain weights for a 28×28 DFT grid.

    mode: 'low'    — Gaussian centred on DC (emphasises low frequencies)
          'high'   — 1 - Gaussian (emphasises high frequencies)
          'band'   — band-pass ring
          'flat'   — all ones (no weighting)
    """
    ky = np.fft.fftfreq(shape[0]) * shape[0]   # -14..13
    kx = np.fft.fftfreq(shape[1]) * shape[1]
    KY, KX = np.meshgrid(ky, kx, indexing='ij')
    R = np.sqrt(KY ** 2 + KX ** 2)             # radial frequency

    if mode == 'low':
        w = np.exp(-R ** 2 / (2 * sigma ** 2))
    elif mode == 'high':
        w = 1.0 - np.exp(-R ** 2 / (2 * sigma ** 2))
    elif mode == 'band':
        center = shape[0] / 4
        w = np.exp(-(R - center) ** 2 / (2 * (sigma / 2) ** 2))
    elif mode == 'flat':
        w = np.ones_like(R)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    return w.flatten()   # shape (784,)


# ============================================================
# 3. Compute DFT features
# ============================================================
print("Computing DFT features (magnitude, log-scale)...")
X_dft = dft_features(X, mode='magnitude')   # (60000, 784)

# Standardize DFT features (zero mean, unit variance per feature)
mu_dft = X_dft.mean(axis=0)
std_dft = X_dft.std(axis=0) + 1e-8
X_dft_norm = (X_dft - mu_dft) / std_dft

print(f"  DFT feature shape: {X_dft_norm.shape}")

# ============================================================
# 4. Pixel-domain vs DFT-domain K-means (K=10)
# ============================================================
K = 10
print(f"\n[Task 2] K-means in pixel domain (K={K})...")
labels_px, C_px, obj_px, _ = kmeans(
    X, K, max_iter=150, n_init=3, init='kmeans++', random_state=42
)
pur_px = cluster_purity(labels_px, y)
nmi_px = normalized_mutual_information(labels_px, y)
print(f"  Pixel — obj={obj_px:.3e}, purity={pur_px:.4f}, NMI={nmi_px:.4f}")

print(f"[Task 2] K-means in DFT domain (K={K})...")
labels_dft, C_dft, obj_dft, _ = kmeans(
    X_dft_norm, K, max_iter=150, n_init=3, init='kmeans++', random_state=42
)
pur_dft = cluster_purity(labels_dft, y)
nmi_dft = normalized_mutual_information(labels_dft, y)
print(f"  DFT   — obj={obj_dft:.3e}, purity={pur_dft:.4f}, NMI={nmi_dft:.4f}")

# ============================================================
# 5. Feature-weighted DFT K-means
# ============================================================
weight_modes = ['flat', 'low', 'high', 'band']
w_results = {}

for wmode in weight_modes:
    w = build_radial_weights(mode=wmode, sigma=6.0)
    # Weights apply to normalised features
    print(f"[Task 2] Weighted DFT K-means — weights={wmode}...")
    labels_w, C_w, obj_w, _ = kmeans(
        X_dft_norm, K, max_iter=150, n_init=3,
        init='kmeans++', w=w, random_state=42
    )
    pur_w = cluster_purity(labels_w, y)
    nmi_w = normalized_mutual_information(labels_w, y)
    w_results[wmode] = dict(labels=labels_w, centroids=C_w,
                            obj=obj_w, purity=pur_w, nmi=nmi_w, w=w)
    print(f"  {wmode:6s} — obj={obj_w:.3e}, purity={pur_w:.4f}, NMI={nmi_w:.4f}")

# ============================================================
# 6. Visualize weight masks
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, wmode in zip(axes, weight_modes):
    w = build_radial_weights(mode=wmode, sigma=6.0)
    # Shift DC to center for visualization
    w_img = np.fft.fftshift(w.reshape(28, 28))
    im = ax.imshow(w_img, cmap='hot', vmin=0)
    ax.set_title(f'Weight mask: {wmode}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('DFT Frequency Weight Masks', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task2_weight_masks.png', dpi=120)
plt.close()
print("\nSaved task2_weight_masks.png")

# ============================================================
# 7. Comparison bar chart
# ============================================================
methods = ['Pixel', 'DFT (flat)'] + [f'DFT ({m})' for m in ['low', 'high', 'band']]
purities = [pur_px, w_results['flat']['purity'],
            w_results['low']['purity'],
            w_results['high']['purity'],
            w_results['band']['purity']]
nmis = [nmi_px, w_results['flat']['nmi'],
        w_results['low']['nmi'],
        w_results['high']['nmi'],
        w_results['band']['nmi']]

x = np.arange(len(methods))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(x, purities, color=['steelblue'] + ['darkorange'] * 4)
ax1.set_xticks(x); ax1.set_xticklabels(methods, rotation=20, ha='right', fontsize=9)
ax1.set_ylabel('Cluster Purity'); ax1.set_title('Purity comparison')
ax1.set_ylim(0, 1); ax1.grid(axis='y', alpha=0.3)

ax2.bar(x, nmis, color=['steelblue'] + ['darkorange'] * 4)
ax2.set_xticks(x); ax2.set_xticklabels(methods, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('NMI'); ax2.set_title('NMI comparison')
ax2.set_ylim(0, 1); ax2.grid(axis='y', alpha=0.3)

plt.suptitle(f'Pixel vs DFT domain K-means (K={K})', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task2_comparison.png', dpi=120)
plt.close()
print("Saved task2_comparison.png")

# ============================================================
# 8. Visualize DFT centroids back in image space
# ============================================================
# Note: centroids in DFT-normalised space; to visualize we un-normalise
# and invert the log-magnitude DFT.

def invert_logmag_dft(log_mag_row, shape=(28, 28)):
    """Approximate image from log|F|. Uses zero phase → symmetric image."""
    F_mag = np.expm1(log_mag_row.reshape(shape)) * (std_dft.reshape(shape) + 1e-8) \
            + mu_dft.reshape(shape)
    F_mag = np.maximum(F_mag, 0)
    # Reconstruct with zero phase
    img = np.fft.ifft2(F_mag).real
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.flat):
    img = invert_logmag_dft(C_dft[k])
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'DFT centroid {k}')
    ax.axis('off')
plt.suptitle('DFT-domain K-means Centroids (K=10, inverted to image space)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task2_dft_centroids.png', dpi=120)
plt.close()
print("Saved task2_dft_centroids.png")

# ============================================================
# Task 2a — Optimizing weights (alternating optimization)
# ============================================================
print("\n[Task 2a] Optimizing feature weights via alternating gradient search")

# Approach: parameterise weights by sigma of Gaussian low-pass filter.
# Scan sigma values (1D grid search), track purity on a validation subset.
# The "held-out" data here is 10 000 images (indices 50k–60k treated as val).

X_tr2a = X_dft_norm[:50000]
y_tr2a = y[:50000]
X_va2a = X_dft_norm[50000:]
y_va2a = y[50000:]

sigmas = [2.0, 4.0, 6.0, 8.0, 11.0, 14.0]
best_sigma = None
best_nmi_va = -1
results_2a = []

for sigma in sigmas:
    w = build_radial_weights(mode='low', sigma=sigma)
    labels_tr_2a, C_2a, _, _ = kmeans(
        X_tr2a, K, max_iter=100, n_init=2,
        init='kmeans++', w=w, random_state=42
    )
    labels_va_2a = assign_clusters(X_va2a, C_2a, w=w)
    purity_va = cluster_purity(labels_va_2a, y_va2a)
    nmi_va = normalized_mutual_information(labels_va_2a, y_va2a)
    obj_va = compute_objective(X_va2a, labels_va_2a, C_2a)
    results_2a.append((sigma, purity_va, nmi_va, obj_va))
    print(f"  sigma={sigma:.1f} → purity_va={purity_va:.4f}, NMI_va={nmi_va:.4f}")
    if nmi_va > best_nmi_va:
        best_nmi_va = nmi_va
        best_sigma = sigma

print(f"\n  Best sigma = {best_sigma} (NMI={best_nmi_va:.4f})")

sigmas_arr, purity_arr, nmi_arr, obj_arr = zip(*results_2a)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(sigmas_arr, purity_arr, 'o-', color='green')
ax1.axvline(best_sigma, linestyle='--', color='red', label=f'best σ={best_sigma}')
ax1.set_xlabel('σ (Gaussian weight bandwidth)'); ax1.set_ylabel('Purity (validation)')
ax1.set_title('Weight Optimisation: Purity vs σ')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(sigmas_arr, nmi_arr, 's-', color='purple')
ax2.axvline(best_sigma, linestyle='--', color='red', label=f'best σ={best_sigma}')
ax2.set_xlabel('σ'); ax2.set_ylabel('NMI (validation)')
ax2.set_title('Weight Optimisation: NMI vs σ')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Task 2a — Grid search over Gaussian frequency weight bandwidth', fontsize=11)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task2a_weight_opt.png', dpi=120)
plt.close()
print("Saved task2a_weight_opt.png")

print("\nTask 2 & 2a complete.")


## 6. Task 3 — Validating Clusters with Train/Test Splits

Fits centroids on training data, evaluates on test data.
Uses purity, NMI, entropy, cluster size imbalance, and objectives.


In [ ]:
"""
Task 3 — Validating clusters with train/test splits.
Task 3a — Different train/test ratios.
"""



# (Data already loaded above)
PLOTS = 'plots'

def dft_features_normalised(images):
    """Return normalised log-magnitude DFT features."""
    N = len(images)
    imgs = images.reshape(N, 28, 28)
    F = np.fft.fft2(imgs)
    feats = np.log1p(np.abs(F)).reshape(N, -1)
    mu = feats.mean(axis=0)
    std = feats.std(axis=0) + 1e-8
    return (feats - mu) / std, mu, std


X_dft, mu_dft, std_dft = dft_features_normalised(X)


def cluster_size_imbalance(labels, K):
    """
    Imbalance metric: max cluster size / mean cluster size.
    1.0 = perfectly balanced; larger = more imbalanced.
    """
    counts = np.bincount(labels, minlength=K).astype(float)
    return counts.max() / (counts.mean() + 1e-10)


# ============================================================
# 2. Task 3 — Main validation (80/20 split)
# ============================================================
K = 10
TRAIN_FRAC = 0.8
N = len(X)
n_train = int(N * TRAIN_FRAC)

rng = np.random.default_rng(0)
perm = rng.permutation(N)
tr_idx = perm[:n_train]
te_idx = perm[n_train:]

X_tr, y_tr = X[tr_idx], y[tr_idx]
X_te, y_te = X[te_idx], y[te_idx]

print(f"\n[Task 3] 80/20 split — train={n_train}, test={N-n_train}")

# --- Pixel domain ---
res_px = train_test_cluster_evaluation(X_tr, X_te, y_tr, y_te,
                                       K=K, n_init=3, random_state=42)
imb_tr_px = cluster_size_imbalance(res_px['train_labels'], K)
imb_te_px = cluster_size_imbalance(res_px['test_labels'], K)

print(f"\nPixel domain (K={K}):")
print(f"  Train obj={res_px['train_obj']:.3e}, test obj={res_px['test_obj']:.3e}")
print(f"  Train purity={res_px['train_purity']:.4f}, test purity={res_px['test_purity']:.4f}")
print(f"  Train NMI={res_px['train_nmi']:.4f}, test NMI={res_px['test_nmi']:.4f}")
print(f"  Train entropy={res_px['train_entropy']:.4f}, test entropy={res_px['test_entropy']:.4f}")
print(f"  Cluster imbalance train={imb_tr_px:.2f}, test={imb_te_px:.2f}")

# --- DFT domain ---
X_dft_tr, X_dft_te = X_dft[tr_idx], X_dft[te_idx]
res_dft = train_test_cluster_evaluation(X_dft_tr, X_dft_te, y_tr, y_te,
                                        K=K, n_init=3, random_state=42)
imb_tr_dft = cluster_size_imbalance(res_dft['train_labels'], K)
imb_te_dft = cluster_size_imbalance(res_dft['test_labels'], K)

print(f"\nDFT domain (K={K}):")
print(f"  Train obj={res_dft['train_obj']:.3e}, test obj={res_dft['test_obj']:.3e}")
print(f"  Train purity={res_dft['train_purity']:.4f}, test purity={res_dft['test_purity']:.4f}")
print(f"  Train NMI={res_dft['train_nmi']:.4f}, test NMI={res_dft['test_nmi']:.4f}")
print(f"  Train entropy={res_dft['train_entropy']:.4f}, test entropy={res_dft['test_entropy']:.4f}")
print(f"  Cluster imbalance train={imb_tr_dft:.2f}, test={imb_te_dft:.2f}")

# --- Comparison figure ---
metrics = ['Purity', 'NMI', 'Entropy', 'Imbalance']
px_tr = [res_px['train_purity'], res_px['train_nmi'],
         res_px['train_entropy'], imb_tr_px]
px_te = [res_px['test_purity'],  res_px['test_nmi'],
         res_px['test_entropy'],  imb_te_px]
dft_tr = [res_dft['train_purity'], res_dft['train_nmi'],
          res_dft['train_entropy'], imb_tr_dft]
dft_te = [res_dft['test_purity'],  res_dft['test_nmi'],
          res_dft['test_entropy'],  imb_te_dft]

x = np.arange(len(metrics))
width = 0.2
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 1.5*width, px_tr,  width, label='Pixel train',  color='steelblue')
ax.bar(x - 0.5*width, px_te,  width, label='Pixel test',   color='cornflowerblue')
ax.bar(x + 0.5*width, dft_tr, width, label='DFT train',    color='darkorange')
ax.bar(x + 1.5*width, dft_te, width, label='DFT test',     color='moccasin')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_title(f'Train vs Test metrics — Pixel and DFT domain (K={K}, 80/20 split)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task3_train_test_metrics.png', dpi=120)
plt.close()
print("\nSaved task3_train_test_metrics.png")

# ============================================================
# 3. Task 3a — Different train/test ratios
# ============================================================
print("\n[Task 3a] Scanning train/test ratios...")
ratios = [0.5, 0.7, 0.8, 0.9]   # fraction of data used for training

records_px  = {'purity_tr':[], 'purity_te':[], 'nmi_tr':[], 'nmi_te':[],
               'obj_tr':[], 'obj_te':[]}
records_dft = {'purity_tr':[], 'purity_te':[], 'nmi_tr':[], 'nmi_te':[],
               'obj_tr':[], 'obj_te':[]}
ratio_labels = []

for frac in ratios:
    n_tr = int(N * frac)
    rng2 = np.random.default_rng(1)
    perm2 = rng2.permutation(N)
    ti = perm2[:n_tr]; vi = perm2[n_tr:]

    lbl = f"{int(frac*100)}/{100-int(frac*100)}"
    ratio_labels.append(lbl)
    print(f"  Ratio {lbl}...", end=' ', flush=True)

    # Pixel
    rp = train_test_cluster_evaluation(X[ti], X[vi], y[ti], y[vi],
                                       K=K, n_init=2, random_state=42)
    records_px['purity_tr'].append(rp['train_purity'])
    records_px['purity_te'].append(rp['test_purity'])
    records_px['nmi_tr'].append(rp['train_nmi'])
    records_px['nmi_te'].append(rp['test_nmi'])
    records_px['obj_tr'].append(rp['train_obj'])
    records_px['obj_te'].append(rp['test_obj'])

    # DFT
    rd = train_test_cluster_evaluation(X_dft[ti], X_dft[vi], y[ti], y[vi],
                                       K=K, n_init=2, random_state=42)
    records_dft['purity_tr'].append(rd['train_purity'])
    records_dft['purity_te'].append(rd['test_purity'])
    records_dft['nmi_tr'].append(rd['train_nmi'])
    records_dft['nmi_te'].append(rd['test_nmi'])
    records_dft['obj_tr'].append(rd['train_obj'])
    records_dft['obj_te'].append(rd['test_obj'])

    print(f"px purity_te={rp['test_purity']:.4f}, "
          f"dft purity_te={rd['test_purity']:.4f}")

# --- Plots ---
x_pos = np.arange(len(ratios))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Purity
axes[0].plot(x_pos, records_px['purity_tr'],  'o--', label='Pixel train',  color='steelblue')
axes[0].plot(x_pos, records_px['purity_te'],  'o-',  label='Pixel test',   color='steelblue', alpha=0.5)
axes[0].plot(x_pos, records_dft['purity_tr'], 's--', label='DFT train',    color='darkorange')
axes[0].plot(x_pos, records_dft['purity_te'], 's-',  label='DFT test',     color='darkorange', alpha=0.5)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(ratio_labels)
axes[0].set_title('Purity vs Train Fraction')
axes[0].set_ylabel('Purity'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# NMI
axes[1].plot(x_pos, records_px['nmi_tr'],  'o--', color='steelblue')
axes[1].plot(x_pos, records_px['nmi_te'],  'o-',  color='steelblue', alpha=0.5)
axes[1].plot(x_pos, records_dft['nmi_tr'], 's--', color='darkorange')
axes[1].plot(x_pos, records_dft['nmi_te'], 's-',  color='darkorange', alpha=0.5)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(ratio_labels)
axes[1].set_title('NMI vs Train Fraction')
axes[1].set_ylabel('NMI'); axes[1].grid(alpha=0.3)
axes[1].legend(['Pixel train','Pixel test','DFT train','DFT test'], fontsize=8)

# Normalised objectives (divide by n_points)
n_tr_list = [int(N * f) for f in ratios]
n_te_list = [N - n for n in n_tr_list]
px_obj_tr_norm  = [o/n for o, n in zip(records_px['obj_tr'],  n_tr_list)]
px_obj_te_norm  = [o/n for o, n in zip(records_px['obj_te'],  n_te_list)]
dft_obj_tr_norm = [o/n for o, n in zip(records_dft['obj_tr'], n_tr_list)]
dft_obj_te_norm = [o/n for o, n in zip(records_dft['obj_te'], n_te_list)]

axes[2].plot(x_pos, px_obj_tr_norm,  'o--', color='steelblue')
axes[2].plot(x_pos, px_obj_te_norm,  'o-',  color='steelblue', alpha=0.5)
axes[2].plot(x_pos, dft_obj_tr_norm, 's--', color='darkorange')
axes[2].plot(x_pos, dft_obj_te_norm, 's-',  color='darkorange', alpha=0.5)
axes[2].set_xticks(x_pos); axes[2].set_xticklabels(ratio_labels)
axes[2].set_title('Objective per point vs Train Fraction')
axes[2].set_ylabel('SSE / N'); axes[2].grid(alpha=0.3)
axes[2].legend(['Pixel train','Pixel test','DFT train','DFT test'], fontsize=8)

plt.suptitle(f'Task 3a — K={K}, varying train/test ratios', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task3a_ratio_comparison.png', dpi=120)
plt.close()
print("Saved task3a_ratio_comparison.png")

# ============================================================
# 4. Stability across multiple random splits (same ratio 80/20)
# ============================================================
print("\n[Task 3a] Stability across 5 random 80/20 splits...")
stab_records_px  = []
stab_records_dft = []

for seed in range(5):
    rng3 = np.random.default_rng(seed * 13)
    perm3 = rng3.permutation(N)
    n_tr3 = int(N * 0.8)
    ti3 = perm3[:n_tr3]; vi3 = perm3[n_tr3:]

    rp3 = train_test_cluster_evaluation(X[ti3], X[vi3], y[ti3], y[vi3],
                                        K=K, n_init=2, random_state=42)
    rd3 = train_test_cluster_evaluation(X_dft[ti3], X_dft[vi3], y[ti3], y[vi3],
                                        K=K, n_init=2, random_state=42)
    stab_records_px.append(rp3['test_purity'])
    stab_records_dft.append(rd3['test_purity'])
    print(f"  Seed {seed}: px purity_te={rp3['test_purity']:.4f}, "
          f"dft purity_te={rd3['test_purity']:.4f}")

print(f"\nPixel  purity std across splits: {np.std(stab_records_px):.5f}")
print(f"DFT    purity std across splits: {np.std(stab_records_dft):.5f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(stab_records_px, 'o-', label=f'Pixel (std={np.std(stab_records_px):.4f})')
ax.plot(stab_records_dft, 's--', label=f'DFT   (std={np.std(stab_records_dft):.4f})')
ax.set_xlabel('Split seed')
ax.set_ylabel('Test purity')
ax.set_title('Stability across 5 random 80/20 splits (K=10)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task3a_stability.png', dpi=120)
plt.close()
print("Saved task3a_stability.png")

print("\nTask 3 & 3a complete.")


## 7. Task 4 — Hierarchical K-Means

Two-level hierarchical K-means: K1=5 coarse clusters, K2=4 sub-clusters each.
Compares with flat K-means at 20 total clusters.


In [ ]:
"""
Task 4 — Hierarchical K-means (2-level).
Compares flat K-means vs hierarchical K-means at similar total leaf clusters.
"""

import time


# (Data already loaded above)
PLOTS = 'plots'
# ============================================================
# 2. Hierarchical K-means implementation
# ============================================================

def hierarchical_kmeans(X, K1, K2, max_iter=100, n_init=3,
                        init='kmeans++', random_state=42):
    """
    2-level hierarchical K-means.

    Level 1: cluster all data into K1 groups.
    Level 2: within each group, run K-means with K2.

    Returns
    -------
    leaf_labels   : ndarray, shape (N,)  — final leaf cluster index (0 .. K1*K2-1)
    level1_labels : ndarray, shape (N,)  — coarse cluster index
    level1_centroids: ndarray, shape (K1, d)
    level2_centroids: list of K1 arrays, each shape (K2, d)
    leaf_obj      : float  — total SSE in leaf assignment
    hierarchy     : dict mapping coarse cluster → sub-cluster info
    """
    N, d = X.shape

    # --- Level 1 ---
    print(f"  Level 1: K={K1}...", end=' ', flush=True)
    t0 = time.time()
    labels1, C1, obj1, _ = kmeans(
        X, K1, max_iter=max_iter, n_init=n_init,
        init=init, random_state=random_state
    )
    print(f"done ({time.time()-t0:.1f}s), obj={obj1:.3e}")

    # --- Level 2 ---
    leaf_labels = np.empty(N, dtype=int)
    leaf_offset = 0
    level2_centroids = []
    hierarchy = {}
    total_obj = 0.0

    for k in range(K1):
        mask = np.where(labels1 == k)[0]
        X_sub = X[mask]
        n_sub = len(X_sub)
        # Adapt K2 if cluster is too small
        k2_actual = min(K2, n_sub)

        print(f"  Level 2, parent={k} (n={n_sub}): K={k2_actual}...",
              end=' ', flush=True)
        t1 = time.time()

        if k2_actual < 2:
            sub_labels = np.zeros(n_sub, dtype=int)
            C_sub = X_sub.mean(axis=0, keepdims=True)
            obj_sub = compute_objective(X_sub, sub_labels, C_sub)
        else:
            sub_labels, C_sub, obj_sub, _ = kmeans(
                X_sub, k2_actual, max_iter=max_iter, n_init=2,
                init=init, random_state=random_state + k
            )
        print(f"done ({time.time()-t1:.1f}s), obj={obj_sub:.3e}")

        total_obj += obj_sub
        level2_centroids.append(C_sub)

        # Map sub-labels to global leaf index
        leaf_labels[mask] = sub_labels + leaf_offset
        hierarchy[k] = {
            'mask': mask,
            'sub_labels': sub_labels,
            'centroids': C_sub,
            'obj': obj_sub,
        }
        leaf_offset += k2_actual

    return (leaf_labels, labels1, C1, level2_centroids,
            total_obj, hierarchy, leaf_offset)


# ============================================================
# 3. Run flat K-means and hierarchical K-means
# ============================================================
K1, K2 = 5, 4     # hierarchical: 5 × 4 = 20 leaf clusters
K_flat = K1 * K2  # flat: 20 clusters

print(f"\n[Task 4] Flat K-means (K={K_flat})...")
t0 = time.time()
labels_flat, C_flat, obj_flat, hist_flat = kmeans(
    X, K_flat, max_iter=150, n_init=3, init='kmeans++', random_state=42
)
t_flat = time.time() - t0
pur_flat = cluster_purity(labels_flat, y)
nmi_flat = normalized_mutual_information(labels_flat, y)
print(f"  Flat K={K_flat}: obj={obj_flat:.3e}, purity={pur_flat:.4f}, "
      f"NMI={nmi_flat:.4f}, t={t_flat:.1f}s")

print(f"\n[Task 4] Hierarchical K-means (K1={K1}, K2={K2})...")
t0 = time.time()
(labels_hier, labels_l1, C_l1, C_l2,
 obj_hier, hierarchy, n_leaves) = hierarchical_kmeans(
    X, K1, K2, max_iter=150, n_init=3, random_state=42
)
t_hier = time.time() - t0
pur_hier = cluster_purity(labels_hier, y)
nmi_hier = normalized_mutual_information(labels_hier, y)
print(f"\n  Hierarchical K={n_leaves} leaves: obj={obj_hier:.3e}, "
      f"purity={pur_hier:.4f}, NMI={nmi_hier:.4f}, t={t_hier:.1f}s")

# ============================================================
# 4. Visualize level-1 centroids
# ============================================================
fig, axes = plt.subplots(1, K1, figsize=(3 * K1, 3))
for k, ax in enumerate(axes):
    ax.imshow(C_l1[k].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    n_k = (labels_l1 == k).sum()
    ax.set_title(f'L1 cluster {k}\n(n={n_k})', fontsize=9)
    ax.axis('off')
plt.suptitle('Hierarchical K-means — Level 1 Centroids', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task4_level1_centroids.png', dpi=120)
plt.close()
print("\nSaved task4_level1_centroids.png")

# ============================================================
# 5. Visualize level-2 centroids for each parent cluster
# ============================================================
fig, axes = plt.subplots(K1, K2, figsize=(2.5 * K2, 2.5 * K1))
for k in range(K1):
    C_sub = hierarchy[k]['centroids']
    for j in range(K2):
        ax = axes[k, j]
        if j < len(C_sub):
            ax.imshow(C_sub[j].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
            ax.set_title(f'L1={k}, L2={j}', fontsize=7)
        ax.axis('off')
plt.suptitle('Hierarchical K-means — Level 2 Centroids', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task4_level2_centroids.png', dpi=120)
plt.close()
print("Saved task4_level2_centroids.png")

# ============================================================
# 6. Comparison: flat vs hierarchical
# ============================================================
metrics = ['Objective (×1e6)', 'Purity', 'NMI', 'Time (s)']
flat_vals = [obj_flat / 1e6, pur_flat, nmi_flat, t_flat]
hier_vals = [obj_hier / 1e6, pur_hier, nmi_hier, t_hier]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, m, fv, hv in zip(axes, metrics, flat_vals, hier_vals):
    ax.bar(['Flat', 'Hier'], [fv, hv],
           color=['steelblue', 'darkorange'], edgecolor='black', alpha=0.85)
    ax.set_title(m)
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate([fv, hv]):
        ax.text(i, v + 0.01 * abs(v), f'{v:.3f}', ha='center', fontsize=9)
plt.suptitle(f'Flat (K={K_flat}) vs Hierarchical ({K1}×{K2}={n_leaves} leaves)',
             fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task4_comparison.png', dpi=120)
plt.close()
print("Saved task4_comparison.png")

# ============================================================
# 7. Digit distribution inside level-1 clusters
# ============================================================
fig, axes = plt.subplots(1, K1, figsize=(3 * K1, 3.5))
for k, ax in enumerate(axes):
    mask = labels_l1 == k
    cnt = np.bincount(y[mask], minlength=10)
    ax.bar(range(10), cnt / cnt.sum(), color='steelblue', alpha=0.8)
    ax.set_xticks(range(10))
    ax.set_xlabel('Digit')
    ax.set_title(f'L1 cluster {k}')
    ax.set_ylabel('Fraction') if k == 0 else None
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Digit distribution in Level-1 clusters', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task4_digit_distribution_l1.png', dpi=120)
plt.close()
print("Saved task4_digit_distribution_l1.png")

# ============================================================
# 8. Additional: K1=10, K2=3 (30 leaves) for deeper hierarchy
# ============================================================
print("\n[Task 4] Extended: K1=10, K2=3 (30 leaves)...")
(labels_30, labels_l1_30, C_l1_30, C_l2_30,
 obj_30, hier_30, n_leaves_30) = hierarchical_kmeans(
    X, 10, 3, max_iter=100, n_init=2, random_state=42
)
pur_30 = cluster_purity(labels_30, y)
nmi_30 = normalized_mutual_information(labels_30, y)
labels_flat30, C_flat30, obj_flat30, _ = kmeans(X, 30, max_iter=100, n_init=2,
                                                init='kmeans++', random_state=42)
pur_flat30 = cluster_purity(labels_flat30, y)
nmi_flat30 = normalized_mutual_information(labels_flat30, y)

print(f"\n  Flat K=30:  obj={obj_flat30:.3e}, purity={pur_flat30:.4f}, NMI={nmi_flat30:.4f}")
print(f"  Hier 10×3: obj={obj_30:.3e}, purity={pur_30:.4f}, NMI={nmi_30:.4f}")

print("\nTask 4 complete.")


## 8. Task 5 — K-Medoids with Edit Distance

Converts images to run-length encoded binary sequences.
Uses Levenshtein edit distance + K-medoids (no centroid averaging).


In [ ]:
"""
Task 5 — K-means with edit distance between images.
Uses K-medoids with edit distance on binary row sequences.
Run on a small subset because edit distance computation is O(N^2 * d).
"""

import time

# (Data already loaded above)
PLOTS = 'plots'
N_EDIT = 500
X_small = X_all[:N_EDIT]
y_small = y_all[:N_EDIT]
print(f"  Using {N_EDIT} images for edit distance clustering.")

# ============================================================
# 2. Convert image to symbolic sequence
# ============================================================

def image_to_sequence(image, threshold=0.3):
    """
    Convert a flattened 28×28 image to a binary sequence.
    Each pixel becomes '1' (ink) or '0' (background) based on threshold.
    Read row by row.

    Returns: list of ints (0 or 1), length 784
    """
    return (image > threshold).astype(np.int8).tolist()


def image_to_rle_sequence(image, threshold=0.3):
    """
    Run-length encoding of binary image (row by row).
    Encode as alternating run lengths starting from 0-pixels.
    This produces a much shorter sequence.

    Returns: list of ints (run lengths)
    """
    binary = (image > threshold).astype(np.int8)
    rle = []
    current = binary[0]
    count = 0
    for b in binary:
        if b == current:
            count += 1
        else:
            rle.append(count)
            count = 1
            current = b
    rle.append(count)
    return rle


# ============================================================
# 3. Edit distance (Levenshtein) between sequences
# ============================================================

def edit_distance(seq1, seq2):
    """
    Standard Levenshtein edit distance between two sequences.
    Uses dynamic programming.
    Cost: insert=1, delete=1, substitute=1.
    """
    m, n = len(seq1), len(seq2)
    # Use 1D DP array for memory efficiency
    dp = np.arange(n + 1, dtype=np.float32)
    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            if seq1[i - 1] == seq2[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1.0 + min(prev, dp[j], dp[j - 1])
            prev = temp
    return float(dp[n])


# ============================================================
# 4. Precompute sequences for all images (RLE for speed)
# ============================================================
print("Converting images to RLE sequences...")
sequences = [image_to_rle_sequence(X_small[i]) for i in range(N_EDIT)]
seq_lengths = [len(s) for s in sequences]
print(f"  Sequence lengths: mean={np.mean(seq_lengths):.1f}, "
      f"min={min(seq_lengths)}, max={max(seq_lengths)}")

# ============================================================
# 5. Compute pairwise edit distance matrix (small subset)
# ============================================================
# For demo, compute full pairwise matrix on N_EDIT images.
print("Computing pairwise edit distances (this may take a minute)...")
t0 = time.time()
D_edit = np.zeros((N_EDIT, N_EDIT), dtype=np.float32)
for i in range(N_EDIT):
    if i % 50 == 0:
        print(f"  Row {i}/{N_EDIT}...", flush=True)
    for j in range(i + 1, N_EDIT):
        d = edit_distance(sequences[i], sequences[j])
        D_edit[i, j] = d
        D_edit[j, i] = d
t_edit = time.time() - t0
print(f"  Done in {t_edit:.1f}s")

# ============================================================
# 6. K-medoids implementation
# ============================================================

def kmedoids(D, K, max_iter=50, n_init=3, random_state=42):
    """
    K-medoids clustering given a precomputed distance matrix D.

    Initialization: random medoids.
    Assignment: each point to nearest medoid.
    Update: find point minimizing total distance to all others in cluster.

    Parameters
    ----------
    D    : ndarray, shape (N, N)  — pairwise distances
    K    : int
    max_iter : int
    n_init   : int
    random_state : int

    Returns
    -------
    best_labels   : ndarray, shape (N,)
    best_medoids  : ndarray, shape (K,)  — indices of medoids
    best_obj      : float  — total distance to medoids
    best_history  : list
    """
    N = D.shape[0]
    rng = np.random.default_rng(random_state)
    best_obj = np.inf
    best_labels = None
    best_medoids = None
    best_history = None

    for run in range(n_init):
        # Initialize: random medoids
        medoids = rng.choice(N, K, replace=False)

        history = []
        for it in range(max_iter):
            # Assignment: argmin distance to medoids
            D_med = D[:, medoids]        # (N, K)
            labels = np.argmin(D_med, axis=1)

            # Objective: sum of distances to assigned medoids
            obj = sum(D[i, medoids[labels[i]]] for i in range(N))
            history.append(obj)

            # Update: new medoid = minimizer of within-cluster sum of distances
            new_medoids = np.empty(K, dtype=int)
            for k in range(K):
                mask = np.where(labels == k)[0]
                if len(mask) == 0:
                    new_medoids[k] = medoids[k]
                    continue
                # Sum of distances within cluster
                D_sub = D[np.ix_(mask, mask)]
                sums = D_sub.sum(axis=1)
                new_medoids[k] = mask[np.argmin(sums)]

            if np.all(new_medoids == medoids):
                break
            medoids = new_medoids

        if obj < best_obj:
            best_obj = obj
            best_labels = labels.copy()
            best_medoids = medoids.copy()
            best_history = history

    return best_labels, best_medoids, best_obj, best_history


# ============================================================
# 7. Run K-medoids with edit distance
# ============================================================
K = 5   # use 5 clusters on 500 images
print(f"\n[Task 5] K-medoids with edit distance (K={K}, N={N_EDIT})...")
t0 = time.time()
labels_med, medoids, obj_med, hist_med = kmedoids(D_edit, K, max_iter=50,
                                                    n_init=3, random_state=42)
t_kmed = time.time() - t0
pur_med = 0.0
pur_med = cluster_purity(labels_med, y_small)
nmi_med = normalized_mutual_information(labels_med, y_small)
print(f"  K-medoids: obj={obj_med:.2f}, purity={pur_med:.4f}, "
      f"NMI={nmi_med:.4f}, t={t_kmed:.1f}s")

# ============================================================
# 8. Compare with Euclidean K-means on same 500 images
# ============================================================
kmeans_eucl = kmeans
print(f"\n[Task 5] Euclidean K-means on same {N_EDIT} images (K={K})...")
labels_eu, C_eu, obj_eu, _ = kmeans_eucl(
    X_small, K, max_iter=100, n_init=3, init='kmeans++', random_state=42
)
pur_eu = cluster_purity(labels_eu, y_small)
nmi_eu = normalized_mutual_information(labels_eu, y_small)
print(f"  K-means:   obj={obj_eu:.4f}, purity={pur_eu:.4f}, NMI={nmi_eu:.4f}")

# ============================================================
# 9. Visualize medoid images
# ============================================================
fig, axes = plt.subplots(1, K, figsize=(3 * K, 3.5))
for k, ax in enumerate(axes):
    med_idx = medoids[k]
    n_k = (labels_med == k).sum()
    ax.imshow(X_small[med_idx].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Medoid {k}\n(n={n_k}, digit {y_small[med_idx]})', fontsize=9)
    ax.axis('off')
plt.suptitle(f'K-medoids with Edit Distance (K={K}, N={N_EDIT})', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task5_medoids.png', dpi=120)
plt.close()
print("\nSaved task5_medoids.png")

# ============================================================
# 10. Convergence of K-medoids
# ============================================================
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_med, 'o-', color='purple')
ax.set_xlabel('Iteration')
ax.set_ylabel('Total distance to medoids')
ax.set_title(f'K-medoids convergence (K={K}, edit distance)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task5_kmedoids_convergence.png', dpi=120)
plt.close()
print("Saved task5_kmedoids_convergence.png")

# ============================================================
# 11. Comparison table: edit distance vs Euclidean
# ============================================================
print("\n=== Task 5 comparison ===")
print(f"{'Method':20s} {'Purity':>8} {'NMI':>8} {'Time(s)':>8}")
print(f"{'K-medoids (edit)':20s} {pur_med:>8.4f} {nmi_med:>8.4f} {t_kmed:>8.1f}")
print(f"{'K-means (Eucl.)':20s} {pur_eu:>8.4f} {nmi_eu:>8.4f} {'<1':>8}")

# --- Bar chart comparison ---
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, vals, title in zip(
    axes,
    [[pur_med, pur_eu], [nmi_med, nmi_eu]],
    ['Cluster Purity', 'NMI']
):
    ax.bar(['K-medoids\n(edit dist)', 'K-means\n(Euclidean)'],
           vals, color=['purple', 'steelblue'], edgecolor='black', alpha=0.85)
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.suptitle(f'Edit distance K-medoids vs Euclidean K-means (K={K}, N={N_EDIT})',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task5_comparison.png', dpi=120)
plt.close()
print("Saved task5_comparison.png")

# ============================================================
# 12. Visualize binary and RLE representations of a few images
# ============================================================
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row_idx, img_idx in enumerate([0, 10, 20]):
    ax = axes[row_idx, 0]
    ax.imshow(X_small[img_idx].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Original (digit {y_small[img_idx]})', fontsize=8)
    ax.axis('off')

    ax = axes[row_idx, 1]
    binary = (X_small[img_idx] > 0.3).reshape(28, 28)
    ax.imshow(binary, cmap='gray', vmin=0, vmax=1)
    ax.set_title('Binary (threshold 0.3)', fontsize=8)
    ax.axis('off')

    ax = axes[row_idx, 2]
    rle = image_to_rle_sequence(X_small[img_idx])
    ax.bar(range(len(rle)), rle, color='steelblue', width=0.8)
    ax.set_title(f'RLE sequence (len={len(rle)})', fontsize=8)
    ax.set_xlabel('Position'); ax.set_ylabel('Run length')

    ax = axes[row_idx, 3]
    seq_full = image_to_sequence(X_small[img_idx])
    ax.imshow(np.array(seq_full).reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title('Binary (flat) = pixel seq', fontsize=8)
    ax.axis('off')

plt.suptitle('Image representations used in edit distance', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/task5_representations.png', dpi=120)
plt.close()
print("Saved task5_representations.png")

print("\nTask 5 complete.")


## Summary

All tasks completed. Plots saved to `plots/` directory.

| Task | Description |
|------|-------------|
| 1    | Basic K-means on MNIST, K ∈ {5,10,15,20,30}, random vs K-means++ |
| 1a   | K-selection: elbow, silhouette, purity, stability |
| 2    | DFT-domain K-means with hand-designed frequency weights |
| 2a   | Gaussian bandwidth grid search for weight optimisation |
| 3    | Train/test validation pipeline with 5 metrics |
| 3a   | Train/test ratio comparison (50/50, 70/30, 80/20, 90/10) |
| 4    | Hierarchical K-means (2-level, K1=5, K2=4) |
| 5    | K-medoids with Levenshtein edit distance on RLE sequences |
